In [3]:
list = [(1, "Hi"), (2, "Bye")]
dict = {1: "Hi", 2: "Bye"}
dict[3] = "Hello"
print(dict)

{1: 'Hi', 2: 'Bye', 3: 'Hello'}


In [5]:
import numpy as np
from mpi4py import MPI
from dolfinx import mesh, fem
import ufl
from petsc4py import PETSc
from my_functions import *
#from visualization_fct import *
from nonlinear_snes_problem import NonlinearPDE_SNESProblem
import pickle
import time
from boundary_condition import BoundaryCondition

In [6]:
1/np.array([1,2,3])

array([1.        , 0.5       , 0.33333333])

In [4]:
def solve_with_Newton(nx, nz, P0, P1, P2, P3, T, layer_params, delta_t = 7, save_tmp=False, filename=None):
    from dolfinx.fem.petsc import create_matrix, create_vector

    # Create FE framework
    domain = create_quad_domain(MPI.COMM_WORLD, nx, nz, P0, P1, P2, P3, celltype=mesh.CellType.quadrilateral)
    Q = fem.functionspace(domain, ("DG", 0)) # material parameters
    V = fem.functionspace(domain, ("CG", 1)) # pressure head
    v = ufl.TestFunction(V)
    x = ufl.SpatialCoordinate(domain)

    # Define delta_t as fem.Constant
    delta_t = fem.Constant(domain, PETSc.ScalarType(delta_t))

    # Assign material properties
    param_fct = assign_material(domain, Q, layer_params)
    Ks = param_fct["Ks"]
    alpha = param_fct["alpha"]
    N = param_fct["N"]
    theta_r = param_fct["theta_r"]
    theta_s = param_fct["theta_s"]

    # Define boundary conditions
    boundaries = {1: lambda x: np.logical_and(np.isclose(x[0], P1[0]), x[1] <= 0), 2: lambda x: np.isclose(x[1], slope*x[0]+P3[1])}
    bc = BoundaryCondition(domain, boundaries)
    c_in = fem.Constant(domain, PETSc.ScalarType(2e-9))
    bc_dict = {1: {"marker": 1, "name": "Dirichlet", "value": lambda x: -x[1], "functionspace": V, "testfunction": v}, 2: {"marker": 2, "name": "Neumann", "value": -c_in, "functionspace": V, "testfunction": v}}
    bc = bc.make_boundary_condition(bc_dict)
    bc_D = bc[1]
    inflow = bc[2]

    t = 0.0 # start time [s]
    
    # Create Newton solver
    snes = PETSc.SNES().create()

    # Variational formulation
    h_w_old = fem.Function(V)
    h_w_old.name = "h_w_old"
    h_w_old.x.array[:] = -1e-2*np.ones_like(h_w_old.x.array) # initial condition close to saturation

    h_w_new = fem.Function(V)

    z = x[1]

    # Weak formulation
    F = (theta(S_e(h_w_new, alpha, N), theta_r, theta_s) - theta(S_e(h_w_old, alpha, N), theta_r, theta_s)) / delta_t * v * ufl.dx
    F += ufl.dot(ufl.grad(v), Ks * k_rel(S_e(h_w_new, alpha, N), N) * ufl.grad(z + h_w_new)) * ufl.dx
    F += inflow

    # Create nonlinear problem
    problem = NonlinearPDE_SNESProblem(F, h_w_new, bc_D)
    b = create_vector(V)
    J = create_matrix(problem.a)

    # Create structure for saving temporary files
    if save_tmp:
        tmp = {
            "nx": nx,
            "nz": nz,
            "T_end": T,
            "alpha": alpha.x.array,
            "N": N.x.array,
            "Ks": Ks.x.array,
            "theta_r": theta_r.x.array,
            "theta_s": theta_s.x.array,
            "h_w": [],
            "times": []
            }
        next_saving_time = 3600
        tmp["h_w"].append(h_w_old.x.array)
        tmp["times"].append(t)

    # Time loop
    while t <= T:

        # Initial guess for Newton
        h_w_new.x.array[:] = h_w_old.x.array
        
        snes.setFunction(problem.F, b) # assemble residual
        snes.setJacobian(problem.J, J) # assemble Jacobian

        # Set options
        snes.setType("newtonls")
        snes.getLineSearch().setType(PETSc.SNESLineSearch.Type.BT)
        snes.setTolerances(rtol=1e-4, atol=1e-11, max_it=20)
        ksp = snes.getKSP()
        ksp.setType("gmres") # iterative solver
        ksp.setTolerances(rtol=1e-4)
        ksp.setErrorIfNotConverged(True)
        ksp.getPC().setType(PETSc.PC.Type.HYPRE)
        ksp.getPC().setHYPREType("boomeramg")
    
        sol_vec = h_w_new.x.petsc_vec.copy() # create solution vector
        sol_vec.ghostUpdate(addv=PETSc.InsertMode.INSERT, mode=PETSc.ScatterMode.FORWARD)
        snes.solve(None, sol_vec) # solve, store solution in solution vector

        sol_vec.copy(h_w_new.x.petsc_vec) # copy solution into h_w_new
        h_w_new.x.scatter_forward()
        
        converged = snes.getConvergedReason()
        num_iter = snes.getIterationNumber()
        
        # adaptive time stepping:
        if num_iter > 10 and float(delta_t.value) > 1e-1:
            delta_t.value = max(0.5*float(delta_t.value), 1e-1)
            continue
        if num_iter < 3 and float(delta_t.value) < 3600:
            delta_t.value = min(float(delta_t.value)*1.2, 3600)
        assert converged > 0, f"Solver did not converge, got {converged}."
        print(
            f"Solver converged after {num_iter} iterations with converged reason {converged}. Time step is {delta_t.value:.2f} s at t={t/3600:.2f} hours."
        )

        # save temporary data
        if save_tmp and t >= next_saving_time:
            next_saving_time += 3600
            tmp["h_w"].append(h_w_new.x.array.copy())
            tmp["times"].append(t)

        # update time
        t += float(delta_t.value)
        # update old solution
        h_w_old.x.array[:] = h_w_new.x.array
    
    # save final data
    if save_tmp:
        tmp["h_w"].append(h_w_new.x.array.copy())
        tmp["times"].append(t)
    # dump temporary data into pickle file
    if save_tmp:
        if filename is None: # set default filename
            filename = "./solutions/heterogeneous_" + str(int(T/3600)) + "h_nx" + str(nx) + "_nz" + str(nz) + ".pkl"
        with open(filename, "wb") as f:
            pickle.dump(tmp, f)
    snes.destroy()
    b.destroy()
    J.destroy()

    return h_w_new
    


# Define geometry
P0 = np.array([0, 0])
P1 = np.array([6, -1])
P2 = np.array([6, 2])
P3 = np.array([0, 3])
slope = (P1[1]-P0[1])/(P1[0]-P0[0])
delta_x = delta_z = 0.1
nx = int(6/delta_x)
nz = int(3/delta_x)
print(f"Resolution is dx = {delta_x} m, dz = {delta_z} m, giving nx = {nx}, nz = {nz}")



# Define parameters for each layer (layer 1, top: sand, layer 2, bottom: silt)
layer_params = {
    1: {"name": "loam", "alpha": 3.6, "N": 1.56, "theta_r": 0.078, "theta_s": 0.43, "Ks": 2.89e-6, "locator": lambda x: False},
    2: {"name": "sand", "alpha": 14.5, "N": 2.68, "theta_r": 0.045, "theta_s": 0.43, "Ks": 8.25e-5, "locator": lambda x: x[1] >= slope*x[0] + P3[1]/2 - 1e-14},
    3: {"name": "silt", "alpha": 1.6, "N": 1.37, "theta_r": 0.034, "theta_s": 0.46, "Ks": 6.94e-7, "locator": lambda x: x[1] < slope*x[0] + P3[1]/2 - 1e-14},
}

T = 24*60*60
t0 = time.time()
h_w = solve_with_Newton(nx, nz, P0, P1, P2, P3, T, layer_params, save_tmp=False)
elapsed = time.time() - t0
print(f"Time needed for execution: {elapsed:.2f} s.")

Resolution is dx = 0.1 m, dz = 0.1 m, giving nx = 60, nz = 30
Solver converged after 1 iterations with converged reason 3. Time step is 8.40 s at t=0.00 hours.
Solver converged after 9 iterations with converged reason 3. Time step is 8.40 s at t=0.00 hours.
Solver converged after 9 iterations with converged reason 3. Time step is 8.40 s at t=0.00 hours.
Solver converged after 8 iterations with converged reason 3. Time step is 8.40 s at t=0.01 hours.
Solver converged after 5 iterations with converged reason 3. Time step is 8.40 s at t=0.01 hours.
Solver converged after 5 iterations with converged reason 3. Time step is 8.40 s at t=0.01 hours.
Solver converged after 5 iterations with converged reason 3. Time step is 8.40 s at t=0.01 hours.
Solver converged after 5 iterations with converged reason 3. Time step is 8.40 s at t=0.02 hours.
Solver converged after 4 iterations with converged reason 3. Time step is 8.40 s at t=0.02 hours.
Solver converged after 5 iterations with converged reaso

In [ ]:
from visualization_fct import *
P0 = np.array([0, 0])
P1 = np.array([6, -1])
P2 = np.array([6, 2])
P3 = np.array([0, 3])
slope = (P1[1]-P0[1])/(P1[0]-P0[0])
delta_x = delta_z = 0.1
nx = int(6/delta_x)
nz = int(3/delta_x)
domain = create_quad_domain(MPI.COMM_WORLD, nx, nz, P0, P1, P2, P3, celltype=mesh.CellType.quadrilateral)
Q = fem.functionspace(domain, ("DG", 0))
param_fct = assign_material(domain, Q, layer_params)
Ks = param_fct["Ks"]
alpha = param_fct["alpha"]
N = param_fct["N"]
theta_r = param_fct["theta_r"]
theta_s = param_fct["theta_s"]

# Evaluate solution
grid, x_plot, z_plot = get_grid(P0, P1, P2, P3, nx, nz)
pressure_head = eval_fct_on_grid(grid, h_w, domain).reshape((nz,nx))
alpha_eval = eval_fct_on_grid(grid, alpha, domain).reshape((nz,nx))
N_eval = eval_fct_on_grid(grid, N, domain).reshape((nz,nx))
theta_r_eval = eval_fct_on_grid(grid, theta_r, domain).reshape((nz,nx))
theta_s_eval = eval_fct_on_grid(grid, theta_s, domain).reshape((nz,nx))
h_tot = pressure_head + grid[:,1].reshape((nz, nx))
h_c = pressure_head * (pressure_head <= 0)
def S_e1(h_c, alpha, N):
    return (1 + (- alpha * h_c)**N)**((1 - N) / N)
theta_tot = theta_r_eval + (theta_s_eval - theta_r_eval)*S_e1(h_c.flatten(), alpha_eval.flatten(), N_eval.flatten()).reshape((nz,nx))

fig, ax = plt.subplots(1,2, layout="constrained", figsize=(14,5))
pmsh1 = ax[0].pcolormesh(x_plot, z_plot, theta_tot, cmap="Blues")
ax[0].set_title(f"LWC after 24 hours")
cbar1 = fig.colorbar(pmsh1, ax=ax[0], shrink=0.5)
cbar1.set_label("theta")

pmsh2 = ax[1].pcolormesh(x_plot, z_plot, h_tot, cmap="Blues")
ax[1].set_title(f"h_tot after 24 hours")
cbar2 = fig.colorbar(pmsh2, ax=ax[1], shrink=0.5)
cbar2.set_label("h_tot [m]")

NameError: name 'create_quad_domain' is not defined

In [3]:
P0 = np.array([0, 0])
P1 = np.array([6, -1])
P2 = np.array([6, 2])
P3 = np.array([0, 3])
slope = (P1[1]-P0[1])/(P1[0]-P0[0])
delta_x = delta_z = 0.1
nx = int(6/delta_x)
nz = int(3/delta_x)
domain = create_quad_domain(MPI.COMM_WORLD, nx, nz, P0, P1, P2, P3)
V = fem.functionspace(domain, ("CG", 1))
v = ufl.TestFunction(V)

c_in = fem.Constant(domain, PETSc.ScalarType(2e-9))
boundaries = {1: lambda x: np.logical_and(np.isclose(x[0], P1[0]), x[1] <= 0), 2: lambda x: np.isclose(x[1], slope*x[0]+P3[1])}
bc_dict = {1: {"marker": 1, "name": "Dirichlet", "value": lambda x: 2, "functionspace": V, "testfunction": v}, 2: {"marker": 2, "name": "Neumann", "value": -c_in, "functionspace": V, "testfunction": v}}
bc = BoundaryCondition(domain, boundaries)
bcs = bc.make_boundary_condition(bc_dict)
bc_D = bcs[1]
inflow = bcs[2]


TypeError: interpolate(): incompatible function arguments. The following argument types are supported:
    1. interpolate(self, f: ndarray[dtype=float64, shape=(*), order='C', writable=False], cells: ndarray[dtype=int32, shape=(*), order='C', writable=False]) -> None
    2. interpolate(self, f: ndarray[dtype=float64, shape=(*, *), order='C', writable=False], cells: ndarray[dtype=int32, shape=(*), order='C', writable=False]) -> None
    3. interpolate(self, u: dolfinx.cpp.fem.Function_float64, cells0: ndarray[dtype=int32, shape=(*), order='C', writable=False], cells1: ndarray[dtype=int32, shape=(*), order='C', writable=False]) -> None
    4. interpolate(self, u: dolfinx.cpp.fem.Function_float64, cells: ndarray[dtype=int32, shape=(*), order='C', writable=False], interpolation_data: dolfinx.cpp.geometry.PointOwnershipData_float64) -> None
    5. interpolate(self, e0: dolfinx.cpp.fem.Expression_float64, cells0: ndarray[dtype=int32, order='C', writable=False], cells1: ndarray[dtype=int32, o

In [7]:
a = lambda x: x[1]
callable(8)

False